In [3]:
# vqc_model.py
# Full VQC for Breast Cancer dataset
# 2 qubits, 4 PCA features, angle encoding, binary classification

import numpy as onp  # standard numpy
import pennylane as qml
from pennylane import numpy as np  # differentiable numpy
from sklearn.metrics import accuracy_score, f1_score

# -------------------------
# 1. Load processed dataset
# -------------------------
def load_breast_cancer_processed():
    """
    Loads the PCA-processed breast cancer data you saved earlier.

    Expects files:
      data/processed/bc_train.npz
      data/processed/bc_val.npz
      data/processed/bc_test.npz

    Each file should contain arrays:
      x: features, shape (N, 4)
      y: labels, 0 or 1
    """
    train = onp.load("data/processed/bc_train.npz")
    val   = onp.load("data/processed/bc_val.npz")
    test  = onp.load("data/processed/bc_test.npz")

    X_train, y_train = train["x"], train["y"]
    X_val,   y_val   = val["x"],   val["y"]
    X_test,  y_test  = test["x"],  test["y"]

    # convert y to float (for loss function)
    y_train = y_train.astype(float)
    y_val   = y_val.astype(float)
    y_test  = y_test.astype(float)

    return X_train, y_train, X_val, y_val, X_test, y_test


# -------------------------
# 2. Quantum device & encoding
# -------------------------
n_qubits = 2      # we are using 2 qubits
n_features = 4    # 4 PCA features
n_layers = 3      # number of variational layers (you can tune this)

# Analytic simulator (no noise, infinite shots)
dev = qml.device("default.qubit", wires=n_qubits, shots=None)

def angle_encode_2q_4f(x):
    """
    Angle encoding for 2 qubits with 4 features.

    x: array-like of shape (4,)

    Mapping:
      - QuBit 0: RY(pi * x0), RZ(pi * x1)
      - QuBit 1: RY(pi * x2), RZ(pi * x3)

    We multiply by pi so that typical feature values map nicely into angle range.
    """
    x = np.array(x).ravel()
    assert len(x) == 4, f"Expected 4 features, got {len(x)}"

    angles = np.pi * x

    # qubit 0 encodes first 2 features
    qml.RY(angles[0], wires=0)
    qml.RZ(angles[1], wires=0)

    # qubit 1 encodes last 2 features
    qml.RY(angles[2], wires=1)
    qml.RZ(angles[3], wires=1)


# -------------------------
# 3. VQC circuit definition
# -------------------------
@qml.qnode(dev)
def vqc_circuit(x, params):
    """
    Quantum circuit for the VQC.

    x: features (4,)
    params: trainable parameters, shape (n_layers, n_qubits, 2)

    For each layer:
      - apply RY, RZ with trainable angles on each qubit
      - then entangle qubits with CNOT(0 -> 1)

    Output: expectation value of PauliZ on qubit 0, in range [-1, 1].
    """
    # 1) Encode classical features into quantum state
    angle_encode_2q_4f(x)

    # 2) Variational layers
    for l in range(n_layers):
        for q in range(n_qubits):
            qml.RY(params[l, q, 0], wires=q)
            qml.RZ(params[l, q, 1], wires=q)

        # entangle qubits
        qml.CNOT(wires=[0, 1])

    # 3) Measure expectation value of Z on qubit 0
    return qml.expval(qml.PauliZ(0))


# -------------------------
# 4. Helper functions: prob, loss, metrics
# -------------------------
def predict_proba_batch(X, params):
    """
    Run the VQC on a batch of inputs X and return probabilities for class 1.

    VQC outputs z in [-1, 1].
    We convert to probability of class 1 as:

        p1 = (1 - z) / 2

    Because:
        z = +1  -> p1 = 0
        z = -1  -> p1 = 1
    """
    outputs = [vqc_circuit(x, params) for x in X]
    outputs = np.stack(outputs)  # shape (N,)

    p1 = (1.0 - outputs) / 2.0
    return p1


def binary_cross_entropy(y_true, p1):
    """
    Standard binary cross-entropy loss.

    y_true: (N,), values 0 or 1
    p1: predicted probability of class 1
    """
    eps = 1e-7
    return -np.mean(y_true * np.log(p1 + eps) + (1 - y_true) * np.log(1 - p1 + eps))


def loss_fn(params, X, y):
    """
    Wrapper loss function for pennylane optimizer.
    """
    p1 = predict_proba_batch(X, params)
    return binary_cross_entropy(y, p1)


def accuracy_from_proba(y_true, p1, threshold=0.5):
    """
    Compute accuracy using probability predictions and a threshold.
    """
    y_pred = (p1 >= threshold).astype(int)
    return np.mean(y_pred == y_true)


# -------------------------
# 5. Training loop
# -------------------------
def train_vqc(
    n_epochs=80,
    lr=0.05,
    print_every=5,
):
    """
    Train the VQC on the breast cancer dataset.

    n_epochs: number of training epochs
    lr: learning rate for Adam optimizer
    print_every: how often to print training status
    """
    # load data
    X_train, y_train, X_val, y_val, X_test, y_test = load_breast_cancer_processed()

    # convert to pennylane numpy arrays
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_val   = np.array(X_val)
    y_val   = np.array(y_val)
    X_test  = np.array(X_test)
    y_test  = np.array(y_test)

    # initialize parameters: small random numbers
    params = 0.01 * np.random.randn(n_layers, n_qubits, 2)

    # optimizer: Adam (better than simple gradient descent)
    opt = qml.AdamOptimizer(stepsize=lr)

    print("Starting VQC training...")
    for epoch in range(1, n_epochs + 1):
        # one optimization step on full training set (no mini-batches here)
        params, train_loss = opt.step_and_cost(
            lambda w: loss_fn(w, X_train, y_train), params
        )

        if (epoch % print_every == 0) or (epoch == 1):
            train_p1 = predict_proba_batch(X_train, params)
            val_p1   = predict_proba_batch(X_val, params)

            train_acc = accuracy_from_proba(y_train, train_p1)
            val_acc   = accuracy_from_proba(y_val,   val_p1)

            print(
                f"Epoch {epoch:3d} | loss={train_loss:.4f} | "
                f"train_acc={float(train_acc):.3f} | val_acc={float(val_acc):.3f}"
            )

    # -------------------------
    # Final metrics (train/val/test, accuracy + F1)
    # -------------------------
    train_p1 = predict_proba_batch(X_train, params)
    val_p1   = predict_proba_batch(X_val,   params)
    test_p1  = predict_proba_batch(X_test,  params)

    y_train_pred = (train_p1 >= 0.5).astype(int)
    y_val_pred   = (val_p1   >= 0.5).astype(int)
    y_test_pred  = (test_p1  >= 0.5).astype(int)

    train_acc = accuracy_score(onp.array(y_train), onp.array(y_train_pred))
    val_acc   = accuracy_score(onp.array(y_val),   onp.array(y_val_pred))
    test_acc  = accuracy_score(onp.array(y_test),  onp.array(y_test_pred))

    train_f1 = f1_score(onp.array(y_train), onp.array(y_train_pred))
    val_f1   = f1_score(onp.array(y_val),   onp.array(y_val_pred))
    test_f1  = f1_score(onp.array(y_test),  onp.array(y_test_pred))

    print("\n===== Final VQC Performance =====")
    print(f"Train Accuracy: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Accuracy:   {val_acc:.4f} | Val F1:   {val_f1:.4f}")
    print(f"Test Accuracy:  {test_acc:.4f} | Test F1:  {test_f1:.4f}")

    return params


# -------------------------
# 6. Main
# ------------------------- 
if __name__ == "__main__":
    # you can change epochs / lr if you want
    train_vqc(n_epochs=80, lr=0.05, print_every=5)


Starting VQC training...
Epoch   1 | loss=1.4378 | train_acc=0.493 | val_acc=0.561
Epoch   5 | loss=1.1688 | train_acc=0.481 | val_acc=0.579
Epoch  10 | loss=0.9830 | train_acc=0.475 | val_acc=0.579
Epoch  15 | loss=0.8486 | train_acc=0.513 | val_acc=0.561
Epoch  20 | loss=0.7676 | train_acc=0.540 | val_acc=0.535
Epoch  25 | loss=0.7293 | train_acc=0.575 | val_acc=0.553
Epoch  30 | loss=0.7044 | train_acc=0.592 | val_acc=0.614
Epoch  35 | loss=0.6859 | train_acc=0.598 | val_acc=0.588
Epoch  40 | loss=0.6758 | train_acc=0.616 | val_acc=0.561
Epoch  45 | loss=0.6723 | train_acc=0.630 | val_acc=0.544
Epoch  50 | loss=0.6711 | train_acc=0.619 | val_acc=0.544
Epoch  55 | loss=0.6706 | train_acc=0.610 | val_acc=0.561
Epoch  60 | loss=0.6694 | train_acc=0.607 | val_acc=0.553
Epoch  65 | loss=0.6693 | train_acc=0.607 | val_acc=0.561
Epoch  70 | loss=0.6694 | train_acc=0.598 | val_acc=0.570
Epoch  75 | loss=0.6692 | train_acc=0.589 | val_acc=0.570
Epoch  80 | loss=0.6689 | train_acc=0.601 | val